## Imports

In [ ]:
from pyspark.sql import SparkSession

In [ ]:

print(SparkSession.builder.getOrCreate().version)

In [ ]:
SparkSession.builder.getOrCreate()

## Create Spark Session

In [ ]:
spark = (
    SparkSession.builder
    .appName("Test")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1"
    )
    .getOrCreate()
)

print(spark.sparkContext.getConf().get("spark.jars.packages"))

## Connect to Kafka

In [ ]:
df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "fraud-transactions")
    .load()
)

## Check Schema

In [ ]:
df.printSchema()

## Convert Messages to String

In [ ]:
from pyspark.sql.functions import col

json_df = df.selectExpr(
    "CAST(value AS STRING)"
)

json_df.printSchema()

## Display Live Stream

In [ ]:
query = (
    json_df.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

## Wait

In [ ]:
query.awaitTermination()

## Run to End the stream

In [ ]:
query.stop()

print("Streaming stopped.")

In [ ]:
spark.streams.active